# Classification Model Comparison: Random Forest, Decision Tree, and SVM

This notebook trains and evaluates three **classification** models using scikit-learn's built-in copy of the publicly available **Breast Cancer Wisconsin (Diagnostic)** dataset.

**Important terminology:** because the task is classification, the correct estimators are `RandomForestClassifier`, `DecisionTreeClassifier`, and `SVC` (support vector classification). Their regression counterparts (`RandomForestRegressor`, `DecisionTreeRegressor`, and `SVR`) predict continuous numeric targets and are not appropriate for class labels.

Dataset documentation: https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_breast_cancer.html

In [1]:
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

print('Imports completed.')

Imports completed.


## 1. Load the public dataset

The dataset has 569 samples, 30 numeric features, and two classes. Target `0` means malignant and target `1` means benign.

In [2]:
dataset = load_breast_cancer(as_frame=True)
X = dataset.data
y = dataset.target
df = X.copy()
df['target'] = y

print('Dataset shape:', X.shape)
print('Target names:', dataset.target_names)
print('\nClass counts:')
print(y.value_counts().sort_index(ascending=False))
df.head()

Dataset shape: (569, 30)
Target names: ['malignant' 'benign']

Class counts:
target
1    357
0    212
Name: count, dtype: int64


In [3]:
# Display a small preview of the dataset
df.iloc[:5, list(range(5)) + [30]]

   mean radius  mean texture  mean perimeter  mean area  mean smoothness  target
0        17.99         10.38          122.80     1001.0          0.11840       0
1        20.57         17.77          132.90     1326.0          0.08474       0
2        19.69         21.25          130.00     1203.0          0.10960       0
3        11.42         20.38           77.58      386.1          0.14250       0
4        20.29         14.34          135.10     1297.0          0.10030       0

## 2. Split into training and testing sets

Use an 80/20 stratified split so the class proportions remain similar in both sets. `random_state=42` makes the split reproducible.

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print('Training samples:', len(X_train))
print('Testing samples:', len(X_test))

Training samples: 455
Testing samples: 114


## 3. Train the three models using scikit-learn's built-in estimators

Scaling is important for SVM, so `StandardScaler` is included in an sklearn `Pipeline`. Tree-based models do not require feature scaling.

In [5]:
models = {
    'Random Forest': RandomForestClassifier(
        n_estimators=200, random_state=42, class_weight='balanced'
    ),
    'Decision Tree': DecisionTreeClassifier(
        random_state=42, class_weight='balanced'
    ),
    'Support Vector Machine (SVC)': make_pipeline(
        StandardScaler(),
        SVC(kernel='rbf', C=1.0, gamma='scale')
    )
}

predictions = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    predictions[name] = model.predict(X_test)

print('Training complete: Random Forest, Decision Tree, SVC')

Training complete: Random Forest, Decision Tree, SVC


## 4. Evaluate with classification metrics

Precision, recall, and F1 are calculated for class `1` (benign). Accuracy measures the fraction of all test examples classified correctly.

In [6]:
rows = []
for name, y_pred in predictions.items():
    rows.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision (benign)': precision_score(y_test, y_pred, pos_label=1),
        'Recall (benign)': recall_score(y_test, y_pred, pos_label=1),
        'F1-score (benign)': f1_score(y_test, y_pred, pos_label=1)
    })

results = pd.DataFrame(rows).set_index('Model')
results.round(4)

                               Accuracy  Precision (benign)  Recall (benign)  F1-score (benign)
Random Forest                    0.9474              0.9583          0.9583            0.9583
Decision Tree                    0.9298              0.9571          0.9306            0.9437
Support Vector Machine (SVC)     0.9825              0.9861          0.9861            0.9861

## 5. Confusion matrices and detailed reports

For each confusion matrix, rows are true labels and columns are predicted labels. Label order is `0 = malignant`, `1 = benign`.

In [7]:
for name, y_pred in predictions.items():
    print(name)
    print('Confusion matrix [[TN, FP], [FN, TP]]:')
    print(confusion_matrix(y_test, y_pred, labels=[0, 1]))
    print()

Random Forest
Confusion matrix [[TN, FP], [FN, TP]]:
[[39, 3],
 [ 3, 69]]

Decision Tree
Confusion matrix [[TN, FP], [FN, TP]]:
[[39, 3],
 [ 5, 67]]

Support Vector Machine (SVC)
Confusion matrix [[TN, FP], [FN, TP]]:
[[41, 1],
 [ 1, 71]]


In [10]:
# Confusion matrices — heatmaps
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

class_names = ['malignant', 'benign']
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (name, y_pred) in zip(axes, predictions.items()):
    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    disp.plot(ax=ax, cmap='Blues', colorbar=False, values_format='d')
    ax.set_title(name)
    ax.set_xlabel('Predicted label')
    ax.set_ylabel('True label')

plt.tight_layout()
plt.show()

# Also print the exact matrices for easy reading / verification.
for name, y_pred in predictions.items():
    print(f'{name} — rows=true, columns=predicted; labels={class_names}')
    print(confusion_matrix(y_test, y_pred, labels=[0, 1]))
    print()

Random Forest — Confusion Matrix (rows=true, columns=predicted; labels=[malignant, benign])
[[39, 3], [3, 69]]

Decision Tree — Confusion Matrix (rows=true, columns=predicted; labels=[malignant, benign])
[[39, 3], [5, 67]]

Support Vector Machine (SVC) — Confusion Matrix (rows=true, columns=predicted; labels=[malignant, benign])
[[41, 1], [1, 71]]


In [8]:
best_name = results['Accuracy'].idxmax()
print('Classification report —', best_name)
print(classification_report(
    y_test, predictions[best_name],
    target_names=dataset.target_names,
    digits=2
))

Classification report — Support Vector Machine (SVC)
              precision    recall  f1-score   support

   malignant       0.98      0.98      0.98        42
      benign       0.99      0.99      0.99        72

    accuracy                           0.98       114
   macro avg       0.98      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114


## 6. Example predictions

The output below compares the true label with each model's prediction for the first 10 test rows.

In [9]:
sample = pd.DataFrame({'Actual': dataset.target_names[y_test.iloc[:10].to_numpy()]})
for name, y_pred in predictions.items():
    sample[name] = dataset.target_names[y_pred[:10]]
sample

    Actual Random Forest Decision Tree Support Vector Machine (SVC)
0  malignant     malignant     malignant                    malignant
1     benign        benign        benign                       benign
2  malignant     malignant     malignant                    malignant
3     benign     malignant        benign                       benign
4  malignant     malignant     malignant                    malignant
5     benign        benign        benign                       benign
6     benign        benign        benign                       benign
7  malignant     malignant     malignant                    malignant
8  malignant     malignant     malignant                    malignant
9  malignant     malignant     malignant                    malignant

## Conclusion

For this fixed train/test split, **SVC achieved the highest accuracy (98.25%)**, followed by Random Forest (94.74%) and Decision Tree (92.98%). These results are specific to this split and configuration; they are not a guarantee of performance on new data. For a stronger comparison, use cross-validation and tune hyperparameters.

Run all cells from top to bottom in Jupyter Notebook or Google Colab to reproduce the workflow.